# Time Series Model Testing on Google Colab

This notebook sets up the environment and tests your trained PatchTST-style temporal transformer model on a new dataset (e.g., sora2_embeddings.h5).


In [ ]:
# Mount Google Drive to access your data files
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# Configure paths - YOUR ACTUAL PATHS
# Path to your test HDF5 file in Google Drive (e.g., sora2_embeddings.h5)
TEST_HDF5_FILE_PATH = "/content/drive/MyDrive/MIT/Lab/sora2_embeddings.h5"

# Path to your trained model checkpoint (.pt file)
CHECKPOINT_PATH = "/content/drive/MyDrive/MIT/Lab/best_model.pt"

# Path to your time_series_model.py file in Google Drive
DRIVE_MODEL_PATH = "/content/drive/MyDrive/MIT/Lab/time_series_model.py"

# Optional: Dataset filter ("sora2", "avdeepfake1m", "shareveo3", or None for all)
DATASET_FILTER = "sora2"  # Set to None to test on all datasets in the file

print("="*60)
print("CONFIGURED PATHS")
print("="*60)
print(f"Test HDF5 file: {TEST_HDF5_FILE_PATH}")
print(f"Checkpoint file: {CHECKPOINT_PATH}")
print(f"Model file (Drive): {DRIVE_MODEL_PATH}")
print(f"Dataset filter: {DATASET_FILTER}")
print("="*60)


In [ ]:
# Copy time_series_model.py from Google Drive
import shutil
import os

if os.path.exists(DRIVE_MODEL_PATH):
    os.makedirs("/content/models", exist_ok=True)
    shutil.copy(DRIVE_MODEL_PATH, "/content/models/time_series_model.py")
    print(f"✓ Copied time_series_model.py from Drive to /content/models/")
    print(f"  Source: {DRIVE_MODEL_PATH}")
    MODEL_FILE = "/content/models/time_series_model.py"
else:
    print(f"⚠️  File not found at {DRIVE_MODEL_PATH}")
    print("Please:")
    print("  1. Upload time_series_model.py to Google Drive")
    print("  2. Update DRIVE_MODEL_PATH in the 'setup_paths' cell above")
    MODEL_FILE = None


In [ ]:
# Install required packages
!pip install h5py numpy scikit-learn tqdm

# Install PyTorch with CUDA support (for GPU)
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118


In [ ]:
# Verify GPU is available
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU device: {torch.cuda.get_device_name(0)}")
    print(f"GPU memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️  WARNING: No GPU detected! Testing will be slower.")
    print("Make sure you've selected a GPU runtime: Runtime > Change runtime type > GPU")


In [ ]:
# Verify test HDF5 file exists and check its structure
import h5py
import os

if os.path.exists(TEST_HDF5_FILE_PATH):
    print(f"✓ Test HDF5 file found at: {TEST_HDF5_FILE_PATH}")
    print(f"  File size: {os.path.getsize(TEST_HDF5_FILE_PATH) / 1e9:.2f} GB")
    
    # Quick check of file structure
    try:
        with h5py.File(TEST_HDF5_FILE_PATH, 'r') as f:
            print("\nFile structure:")
            def print_structure(name, obj):
                print(f"  {name}: {type(obj).__name__}")
            f.visititems(print_structure)
            
            # Check for videos group
            if 'videos' in f:
                num_videos = len(f['videos'])
                print(f"\n  Number of videos: {num_videos}")
                
                # Check first video for dataset attribute
                if num_videos > 0:
                    first_vid_key = list(f['videos'].keys())[0]
                    first_vid = f['videos'][first_vid_key]
                    if 'dataset' in first_vid.attrs:
                        dataset_attr = first_vid.attrs['dataset']
                        if isinstance(dataset_attr, bytes):
                            dataset_attr = dataset_attr.decode()
                        print(f"  First video dataset: {dataset_attr}")
                        
                    # Check embeddings
                    if 'embeddings' in first_vid:
                        emb_grp = first_vid['embeddings']
                        print(f"  Available embeddings: {list(emb_grp.keys())}")
                        
                        # Check dimensions
                        if 'openl3' in emb_grp:
                            print(f"    openl3 shape: {emb_grp['openl3'].shape}")
                        if 'senet' in emb_grp:
                            print(f"    senet shape: {emb_grp['senet'].shape}")
    except Exception as e:
        print(f"  ⚠️  Error reading file: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⚠️  Test HDF5 file NOT found at: {TEST_HDF5_FILE_PATH}")
    print("\nPlease:")
    print("  1. Upload your sora2_embeddings.h5 file to Google Drive")
    print("  2. Update TEST_HDF5_FILE_PATH in the 'setup_paths' cell above")


In [ ]:
# Verify checkpoint file exists
import os

if os.path.exists(CHECKPOINT_PATH):
    print(f"✓ Checkpoint file found at: {CHECKPOINT_PATH}")
    file_size = os.path.getsize(CHECKPOINT_PATH) / 1e6  # MB
    print(f"  File size: {file_size:.2f} MB")
    
    # Load and inspect checkpoint
    try:
        import torch
        checkpoint = torch.load(CHECKPOINT_PATH, map_location='cpu', weights_only=False)
        
        print("\nCheckpoint info:")
        print(f"  Keys: {list(checkpoint.keys())}")
        if 'epoch' in checkpoint:
            print(f"  Epoch: {checkpoint['epoch']}")
        if 'val_auroc' in checkpoint:
            print(f"  Validation AUROC: {checkpoint['val_auroc']:.4f}")
        if 'model_state_dict' in checkpoint:
            num_params = sum(p.numel() for p in checkpoint['model_state_dict'].values())
            print(f"  Model parameters: {num_params:,}")
    except Exception as e:
        print(f"  ⚠️  Error reading checkpoint: {e}")
else:
    print(f"⚠️  Checkpoint file NOT found at: {CHECKPOINT_PATH}")
    print("\nPlease:")
    print("  1. Upload your best_model.pt checkpoint to Google Drive")
    print("  2. Update CHECKPOINT_PATH in the 'setup_paths' cell above")


In [ ]:
# IMPORTANT: Check label distribution in sora2 dataset before testing
# This helps diagnose AUROC=0.0 issues
import h5py
import numpy as np

print("="*60)
print("CHECKING LABEL DISTRIBUTION IN TEST DATASET")
print("="*60)

if os.path.exists(TEST_HDF5_FILE_PATH):
    try:
        with h5py.File(TEST_HDF5_FILE_PATH, 'r') as f:
            if 'videos' not in f:
                print("⚠️  No 'videos' group found in HDF5 file")
            else:
                videos_grp = f['videos']
                video_keys = list(videos_grp.keys())
                print(f"Total videos in file: {len(video_keys)}")
                
                # Check first few videos for label distribution
                all_labels_sample = []
                datasets_found = set()
                videos_checked = 0
                max_videos_to_check = 50
                
                for vid_key in video_keys[:max_videos_to_check]:
                    vid_grp = videos_grp[vid_key]
                    
                    # Check dataset attribute
                    if 'dataset' in vid_grp.attrs:
                        dataset_attr = vid_grp.attrs['dataset']
                        if isinstance(dataset_attr, bytes):
                            dataset_attr = dataset_attr.decode()
                        datasets_found.add(dataset_attr.lower())
                        
                        # Only check videos matching our filter (if specified)
                        if DATASET_FILTER:
                            if DATASET_FILTER.lower() not in dataset_attr.lower():
                                continue
                    
                    # Check if we have labels
                    if 'labels' not in vid_grp:
                        continue
                    
                    labels_grp = vid_grp['labels']
                    label_type = 'audio' if 'audio' in labels_grp else ('video' if 'video' in labels_grp else None)
                    
                    if label_type:
                        labels_data = labels_grp[label_type][:]  # Shape: [num_augs, num_segs]
                        if len(labels_data) > 0:
                            # Take first augmentation as sample
                            all_labels_sample.extend(labels_data[0].flatten())
                            videos_checked += 1
                
                if len(all_labels_sample) > 0:
                    all_labels_sample = np.array(all_labels_sample)
                    unique_labels, counts = np.unique(all_labels_sample, return_counts=True)
                    
                    print(f"\n📊 Label Distribution (from {videos_checked} videos, {len(all_labels_sample)} segments):")
                    print(f"   Datasets found: {list(datasets_found)}")
                    
                    for label_val, count in zip(unique_labels, counts):
                        pct = 100 * count / len(all_labels_sample)
                        # Determine if label is real or fake
                        if label_val > 0.5:
                            label_name = "REAL"
                            binary_val = 1
                        else:
                            label_name = "FAKE"
                            binary_val = 0
                        print(f"   {label_name} (value={label_val:.3f}, binary={binary_val}): {count:,} segments ({pct:.2f}%)")
                    
                    # Check if we have both classes
                    binary_labels = (all_labels_sample > 0.5).astype(int)
                    unique_binary = np.unique(binary_labels)
                    
                    print(f"\n   Binary label classes present: {unique_binary.tolist()}")
                    
                    if len(unique_binary) == 1:
                        print("\n   ⚠️  WARNING: Only ONE class present in labels!")
                        print(f"      All labels are: {'REAL' if unique_binary[0] == 1 else 'FAKE'}")
                        print("      → AUROC will be 0.0 because AUROC requires both classes")
                        print("      → This is expected if sora2 dataset only contains one class")
                        print("      → You can still check Accuracy and Loss metrics")
                    else:
                        print("   ✓ Both classes present - AUROC should be calculable")
                else:
                    print("⚠️  No labels found in sampled videos")
                    
    except Exception as e:
        print(f"⚠️  Error checking labels: {e}")
        import traceback
        traceback.print_exc()
else:
    print(f"⚠️  Test HDF5 file not found: {TEST_HDF5_FILE_PATH}")


In [ ]:
# Run the test script
import subprocess
import sys
import os

if MODEL_FILE and os.path.exists(MODEL_FILE):
    if os.path.exists(TEST_HDF5_FILE_PATH) and os.path.exists(CHECKPOINT_PATH):
        print("="*60)
        print("Starting Model Testing")
        print("="*60)
        print(f"Model file: {MODEL_FILE}")
        print(f"Test HDF5 file: {TEST_HDF5_FILE_PATH}")
        print(f"Checkpoint: {CHECKPOINT_PATH}")
        if DATASET_FILTER:
            print(f"Dataset filter: {DATASET_FILTER}")
        print("="*60 + "\n")
        
        # Build command
        cmd = [
            sys.executable,
            MODEL_FILE,
            "test",
            TEST_HDF5_FILE_PATH,
            CHECKPOINT_PATH,
        ]
        
        if DATASET_FILTER:
            cmd.append(DATASET_FILTER)
        
        # Run the test script
        result = subprocess.run(
            cmd,
            capture_output=False,
            text=True
        )
        
        if result.returncode == 0:
            print("\n✅ Testing completed successfully!")
        else:
            print(f"\n⚠️  Testing exited with code {result.returncode}")
    else:
        missing = []
        if not os.path.exists(TEST_HDF5_FILE_PATH):
            missing.append(f"Test HDF5 file: {TEST_HDF5_FILE_PATH}")
        if not os.path.exists(CHECKPOINT_PATH):
            missing.append(f"Checkpoint: {CHECKPOINT_PATH}")
        print("⚠️  Cannot run testing - missing files:")
        for item in missing:
            print(f"  - {item}")
else:
    print("⚠️  Cannot run testing - model file not found")
    print("Please ensure time_series_model.py is available")


In [ ]:
# Alternative: Test using Python import (more interactive)
# This allows you to inspect results and metrics in more detail

import sys
import os

# Add model directory to path
sys.path.insert(0, '/content/models')

if MODEL_FILE and os.path.exists(MODEL_FILE):
    if os.path.exists(TEST_HDF5_FILE_PATH) and os.path.exists(CHECKPOINT_PATH):
        print("="*60)
        print("Testing Model (Interactive Mode)")
        print("="*60)
        
        # Import the test function
        from time_series_model import test_main
        
        # Run test
        test_metrics = test_main(
            hdf5_path=TEST_HDF5_FILE_PATH,
            checkpoint_path=CHECKPOINT_PATH,
            filter_dataset=DATASET_FILTER,
            audio_embedding_type="openl3",
            video_embedding_type="senet",
            use_audio_labels=True,
            batch_size=16,
        )
        
        print("\n" + "="*60)
        print("FINAL TEST RESULTS")
        print("="*60)
        print(f"Test Loss: {test_metrics['loss']:.4f}")
        print(f"Test AUROC: {test_metrics['auroc']:.4f}")
        print(f"Test Accuracy: {test_metrics['accuracy']:.4f}")
        print("="*60)
    else:
        print("⚠️  Missing required files. Please check the paths above.")
else:
    print("⚠️  Model file not found. Please check DRIVE_MODEL_PATH.")


## Notes

1. **GPU Runtime**: Make sure you've selected a GPU runtime (Runtime > Change runtime type > GPU) for faster testing
2. **HDF5 File**: Upload your `sora2_embeddings.h5` file to Google Drive and update `TEST_HDF5_FILE_PATH`
3. **Checkpoint**: Upload your trained `best_model.pt` checkpoint to Google Drive and update `CHECKPOINT_PATH`
4. **Model File**: Upload `time_series_model.py` to Google Drive and update `DRIVE_MODEL_PATH`
5. **Dataset Filter**: Set `DATASET_FILTER` to "sora2" to test only on sora2 data, or `None` to test on all datasets

## Setup Checklist

- [ ] Mounted Google Drive
- [ ] Updated `TEST_HDF5_FILE_PATH` to your test HDF5 file location in Drive
- [ ] Updated `CHECKPOINT_PATH` to your trained model checkpoint location in Drive
- [ ] Updated `DRIVE_MODEL_PATH` to your `time_series_model.py` location in Drive
- [ ] Set `DATASET_FILTER` appropriately (e.g., "sora2" or None)
- [ ] Selected GPU runtime (Runtime > Change runtime type > GPU)
- [ ] Run all cells in order
